# WP3 — Partie 1 : Génération des paires (z_full, z_clip)

### 1.1 Chargement MAE

In [1]:
import os
import torch
from transformers import ViTImageProcessor, ViTMAEModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

MAE_PATH = './vit-mae-large' if os.path.exists('./vit-mae-large') else 'facebook/vit-mae-large'
print(f'Chargement MAE depuis : {MAE_PATH}')

mae_processor = ViTImageProcessor.from_pretrained(MAE_PATH)
mae_encoder = ViTMAEModel.from_pretrained(MAE_PATH).to(DEVICE)
mae_encoder.eval()
print('MAE chargé')

Device : cpu
Chargement MAE depuis : facebook/vit-mae-large


/Users/Ahmed/miniconda3/envs/torch/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MAE chargé


### 1.2 Chargement CLIP

In [2]:
import os
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

CLIP_PATH = './clip-vit-large-patch14' if os.path.exists('./clip-vit-large-patch14') else 'openai/clip-vit-large-patch14'
print(f'Chargement CLIP depuis : {CLIP_PATH}')

clip_model = CLIPModel.from_pretrained(CLIP_PATH).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_PATH)
clip_model.eval()

CLIP_DIM = clip_model.config.projection_dim
MAE_DIM = mae_encoder.config.hidden_size
print(f'MAE dim  : {MAE_DIM}')   # 1024
print(f'CLIP dim : {CLIP_DIM}')  # 768

Chargement CLIP depuis : openai/clip-vit-large-patch14
MAE dim  : 1024
CLIP dim : 768


### 1.3 Chargement dataset

In [3]:
import os
from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision import transforms
from huggingface_hub import snapshot_download

DATASET_DIR = './imagenet100-hf'
if not os.path.exists(f'{DATASET_DIR}/data'):
    print('Téléchargement ImageNet-100...')
    snapshot_download(repo_id='ilee0022/ImageNet100', repo_type='dataset', local_dir=DATASET_DIR)
    print('Dataset téléchargé')

transform_tensor = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),  # float32 dans [0, 1]
])

class ImageNet100Dataset(Dataset):
    def __init__(self, split, transform=None):
        pattern = f'{DATASET_DIR}/data/{split}-*.parquet'
        self.ds = load_dataset('parquet', data_files={split: pattern})[split]
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img = self.ds[idx]['image'].convert('RGB')
        label = self.ds[idx]['label']
        if self.transform:
            img = self.transform(img)
        return img, label

dataset_train = ImageNet100Dataset('train', transform=transform_tensor)
dataset_val = ImageNet100Dataset('validation', transform=transform_tensor)

print(f'Train : {len(dataset_train)} images')
print(f'Val   : {len(dataset_val)} images')

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

Train : 117000 images
Val   : 13000 images


### 1.4 Génération des paires en batch

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def generate_pairs(dataset, batch_size=64, desc=''):
    """
    Genere (z_full, z_clip) pour chaque image en mode batch.
    do_rescale=False car ToTensor a deja mis les pixels dans [0, 1].
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == 'cuda'),
    )

    all_z_full, all_z_clip, all_labels = [], [], []

    for images, labels in tqdm(loader, desc=desc):
        B = images.shape[0]

        # --- MAE ---
        mae_inputs = mae_processor(
            images=list(images), return_tensors='pt', do_rescale=False
        )
        mae_inputs = {k: v.to(DEVICE) for k, v in mae_inputs.items()}
        with torch.no_grad():
            out = mae_encoder(
                **mae_inputs,
                noise=torch.zeros(B, 196).to(DEVICE)
            )
        z_full = out.last_hidden_state[:, 1:].mean(dim=1).cpu()  # (B, 1024)

        # --- CLIP ---
        clip_inputs = clip_processor(
            images=list(images), return_tensors='pt', do_rescale=False
        )
        clip_inputs = {k: v.to(DEVICE) for k, v in clip_inputs.items()}
        with torch.no_grad():
            vision_out = clip_model.vision_model(**clip_inputs)
            z_clip = clip_model.visual_projection(vision_out.pooler_output)
        z_clip = F.normalize(z_clip, dim=-1).cpu()  # (B, 768)

        all_z_full.append(z_full)
        all_z_clip.append(z_clip)
        all_labels.append(labels)

    return {
        'z_full': torch.cat(all_z_full),
        'z_clip': torch.cat(all_z_clip),
        'labels': torch.cat(all_labels),
    }

### 1.5 Lancement et sauvegarde

In [ ]:
data_train = generate_pairs(dataset_train, batch_size=64, desc='Train')
data_val = generate_pairs(dataset_val, batch_size=64, desc='Val')

torch.save(data_train, 'pairs_train.pt')
torch.save(data_val, 'pairs_val.pt')

print('Sauvegarde OK')
print(f'  Train z_full : {data_train["z_full"].shape}')   # (N, 1024)
print(f'  Train z_clip : {data_train["z_clip"].shape}')   # (N, 768)
print(f'  Val   z_full : {data_val["z_full"].shape}')

# Liberer la VRAM — important sur serveur partage
del mae_encoder, clip_model
torch.cuda.empty_cache()
print('VRAM liberee')